In [1]:
import warnings; warnings.filterwarnings("ignore")
import time
import jax
jax.config.update("jax_platform_name", "gpu")
from jaxmat.tensors import SymmetricTensor2, dev, main_invariants
from jaxmat.state import AbstractState
from pathlib import Path
import jax.numpy as jnp
import equinox as eqx
import optax
import matplotlib
from jaxmat.utils import default_value
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from jaxmat.nn.icnn import ICNN, QuadICNN, ICNNSkip
import jaxmat.materials as jm
from jaxmat.loader import ImposedLoading, global_solve
from jaxmat.materials.viscoplastic_flows import (
    ArmstrongFrederickHardening, NortonFlow, VoceHardening,
)
from jaxmat.materials.viscoplasticity import ArmstrongFrederickViscoplasticity
from jaxmat.utils import partition_by_node_names
 
 


In [3]:
TRUE = dict(
    E=196_000., nu=0.3,
    sig0=150., sigu=300., b=15.,
    C1=50_000., gamma1=300.,
    C2=20_000., gamma2=100.,
    K=10., m=8.,
)

TRUE_PLAST = dict(H_max = 20e3,E=196_000., nu=0.3,sig0=300,H=5e3)

class LinearHardening(eqx.Module):
    sig0: jax.Array = eqx.field(converter=lambda x: jnp.asarray(x, dtype=jnp.float64))  # Limite d'élasticité (MPa)
    H: jax.Array = eqx.field(converter=lambda x: jnp.asarray(x, dtype=jnp.float64))      # Module d'écrouissage (MPa)
    
    def __call__(self, p):
        return self.sig0 + self.H * p



def make_material_plast(p):
    elasticity = jm.LinearElasticIsotropic(E=p["E"], nu=p["nu"])
    hardening = LinearHardening(sig0=p["sig0"], H=p["H"])
    return jm.vonMisesIsotropicHardening(elasticity=elasticity, yield_stress=hardening)

def make_material(p):
    return ArmstrongFrederickViscoplasticity(
        elasticity=jm.LinearElasticIsotropic(E=p["E"], nu=p["nu"]),
        yield_stress=VoceHardening(sig0=p["sig0"], sigu=p["sigu"], b=p["b"]),
        kinematic_hardening=ArmstrongFrederickHardening(
            C    =jnp.array([p["C1"],     p["C2"]]),
            gamma=jnp.array([p["gamma1"], p["gamma2"]]),
        ),
        viscous_flow=NortonFlow(K=p["K"], m=p["m"]),
    )
 
material_true = make_material(TRUE)
# material_true = make_material_plast(TRUE_PLAST)

 


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.  CHARGEMENT : 2 cycles ±1.5 %, 60 pas/demi-cycle
# ═══════════════════════════════════════════════════════════════════════════════
eps_max  = 0.015
n_cycles = 1
Nsteps   = 60
 
half     = jnp.linspace(0, eps_max, Nsteps + 1)
cycle    = jnp.concatenate([half, half[::-1][1:], -half[1:], -half[::-1][1:]])
eps_path = jnp.tile(cycle[1:], n_cycles)
dt_val   = float(eps_max / (1e-3 * Nsteps))
print(f"Chargement : {len(eps_path)} pas, {n_cycles} cycles, dt = {dt_val:.3f} s")
 
# 

Chargement : 240 pas, 1 cycles, dt = 0.250 s


In [5]:
from optax.tree_utils import tree_scale, tree_sub
# define evolution function 
@eqx.filter_jit
def compute_evolution(material, epsilons_xx, times, return_potential=False):
    # Initial material state
    state0 = material.init_state()
    dt = jnp.diff(times)

    def step(state, args):
        eps_xx, dt_ = args


        # Update material response
        sig_new, new_state = material.constitutive_update_mixed_control(eps_xx, state, dt_)

        if return_potential:
            # Free energy at n+1
            free_energy = material.free_energy(eps_xx, new_state.internal)

            # Internal variable rate
            d_isv = tree_sub(new_state.internal, state.internal)
            isv_dot = tree_scale(1.0 / dt_, d_isv)

            # Dissipative thermodynamic force
            A_dis = jax.grad(material.free_energy, argnums=1)(
                eps_xx, new_state.internal
            )
            A_dis = tree_scale(-1,A_dis)

            # Dissipation: A_dis · isv_dot
            dissipation = jnp.sum(A_dis.alpha * isv_dot.alpha) 

            return new_state, (
                sig_new,
                free_energy,
                dissipation,
                isv_dot,
                new_state.internal,
            )
        else:
            return new_state, new_state

    if return_potential:
        _, (sig, free_energy, dissipation, isv_dot, isv) = jax.lax.scan(
            step, state0, (epsilons_xx, dt)
        )
        return sig, free_energy, dissipation, isv_dot, isv
    else:
        _, states = jax.lax.scan(step, state0, (epsilons_xx, dt))
        return states

batched_compute_evolution = jax.vmap(
    compute_evolution,
    in_axes=(None, 0, 0, None),
)

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.  INTÉGRATION DIFFÉRENTIABLE (lax.scan)
# ═══════════════════════════════════════════════════════════════════════════════
@eqx.filter_jit
def compute_jit(material, eps_path, dt):
    """Intègre la loi sur le chemin eps_path (chargement mixte).
    Retourne σ_xx à chaque pas. Différentiable par rapport aux paramètres."""
    state0 = material.init_state(1)
    def step(carry, eps_xx_val):
        Eps, state = carry
        loading = ImposedLoading(
            epsxx=jnp.ones(1) * eps_xx_val,
            sigyy=jnp.zeros(1),
            sigzz=jnp.zeros(1),
        )
        Eps_new, state_new, _ = global_solve(Eps, state, loading, material, dt)
        return (Eps_new, state_new), state_new.stress[0, 0, 0]
    _, sig = jax.lax.scan(step, (state0.strain, state0), eps_path)
    return sig

@eqx.filter_jit
def compute_state_jit(material, eps_path, dt):
    """Intègre la loi sur le chemin eps_path (chargement mixte).
    Retourne σ_xx à chaque pas. Différentiable par rapport aux paramètres."""
    state0 = material.init_state(1)
    def step(carry, eps_xx_val):
        Eps, state = carry
        loading = ImposedLoading(
            epsxx=jnp.ones(1) * eps_xx_val,
            sigyy=jnp.zeros(1),
            sigzz=jnp.zeros(1),
        )
        Eps_new, state_new, _ = global_solve(Eps, state, loading, material, dt)
        return (Eps_new, state_new), state_new
    _, sig = jax.lax.scan(step, (state0.strain, state0), eps_path)
    return sig
 
 


In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4.  DONNÉES SYNTHÉTIQUES (vraie réponse + bruit 2 %)
# ═══════════════════════════════════════════════════════════════════════════════
print("Génération des données de référence...")
sig_clean = compute_jit(material_true, eps_path, dt_val)
key       = jax.random.PRNGKey(42)
noise_lvl = 0.02
sig_noisy = sig_clean #+ jax.random.normal(key, sig_clean.shape) * \
            # jnp.max(jnp.abs(sig_clean)) * noise_lvl 
sig_mean = jnp.abs(sig_clean).mean()
sig_noisy_norm =sig_noisy /  sig_mean 
print(f"  σ_xx ∈ [{float(sig_noisy.min()):.1f}, {float(sig_noisy.max()):.1f}] MPa")
print(f"  sig_noisy_norm ∈ [{float(sig_noisy_norm.min()):.1f}, {float(sig_noisy_norm.max()):.1f}]")

eps_mean = jnp.abs(eps_path).mean()
eps_path_norm = eps_path / eps_mean

Génération des données de référence...
  σ_xx ∈ [-555.1, 484.3] MPa
  sig_noisy_norm ∈ [-1.6, 1.4]


In [9]:

# ═══════════════════════════════════════════════════════════════════════════════
# 5.  MODÈLE INITIAL (gsm)
# ═══════════════════════════════════════════════════════════════════════════════
class InternalState(AbstractState):
    """
    Internal state for hardening viscoplasticity
    """

    p: jax.Array = eqx.field(init=False)
    """scalar """
    epsp: SymmetricTensor2 = eqx.field(default_factory=lambda: SymmetricTensor2())
    """Plastic strain tensor"""
    Nvar: int = eqx.field(static=True, default=1)

    def __post_init__(self):
        # Initialise to zero
        self.p = jnp.zeros((self.Nvar,))

InternalState = jm.elastoplasticity.InternalState


# int_var = InternalState(Nvar=1)


# class FreeEnergy(eqx.Module):
#     elasticity: jm.AbstractLinearElastic  # Hooke isotrope
#     nn_anelastic: ICNN  # ICNN for epsp variable
#     nn_hardening: ICNN  # ICNN for p variable

#     def __call__(self, eps, isv):
#         """
#         Compute free energy:
#         psi = 0.5*(eps - sum_i alpha_p[i]) : C : (eps - sum_i alpha_p[i]) + NN(epsp) + NN(q)
#         """
#         epsp = isv.epsp
#         psi_elastic = 0.5 * jnp.trace((eps - epsp) @ (self.elasticity.C @ (eps - epsp)))

#         _, I2, _ = main_invariants(isv.epsp)
#         psi_anelastic = self.nn_anelastic(I2) - self.nn_anelastic(jnp.zeros_like(I2)) # - jax.jvp(self.nn_anelastic,(jnp.zeros_like(I2),),(I2,))[1]
#         psi_hardening = self.nn_hardening(isv.p)  - self.nn_hardening(jnp.zeros_like(isv.p)) #- jax.jvp(self.nn_hardening,(jnp.zeros_like(isv.p),),(isv.p,))[1]
#         # énergie totale
#         return psi_elastic + psi_hardening + psi_anelastic 

class FreeEnergy(eqx.Module):
    elasticity: jm.AbstractLinearElastic   # Hooke isotrope
    nn_hardening: ICNN                # ICNN pour l'écrouissage

    def __call__(self, eps, isv):
        """
        Compute free energy:
        psi = 0.5*(eps - epsp[i]) : C : (eps - epsp[i]) + NN(q)
        """
        psi_elastic = 0.5 * jnp.trace((eps - isv.epsp) @ (self.elasticity.C @ (eps - isv.epsp)))

        # --- Partie écrouissage ---
        # q doit être de forme (Nvar,)
        psi_hardening = self.nn_hardening(isv.p) - self.nn_hardening(jnp.zeros_like(isv.p)) - jax.jvp(
            self.nn_hardening,
            (jnp.zeros_like(isv.p),),
            (isv.p,),
        )[1]

        # énergie totale
        return psi_elastic + psi_hardening

from jaxmat.tensors import SymmetricTensor2, dev, eigenvalues
from jaxmat.tensors.utils import safe_norm, safe_sqrt

def safe_zero(method):
    """Decorator for yield surfaces to avoid NaNs for zero stress in both fwd and bwd AD."""

    def wrapper(self, x, *args, eps=1e-16):
        x_norm = jnp.linalg.norm(x)
        x_safe = SymmetricTensor2(tensor=jnp.where(x_norm > eps, x, 0 * x))
        return jnp.where(x_norm > eps, method(self, x_safe, *args), 0.0)

    return wrapper
class ICNNDissipationPotential(QuadICNN):
    @safe_zero
    def von_mises(self, sig):
        return jnp.sqrt(3 / 2.0) * safe_norm(dev(sig))
    def icnn_potential(self, A, Q=None):
        """
        A : SymmetricTensor2 (batch Nvar, 3,3)
        Q : jnp.ndarray (batch Nvar,) ou None
        """
        # Calcul invariants principaux du batch de A
        # I1, I2, I3 = jax.vmap(main_invariants)(A)  # shape (Nvar,)
        I1, I2, I3 = main_invariants(A)
        I2 = self.von_mises(A)
        # I2 = jnp.sqrt(3 / 2.0) * safe_norm(A - SymmetricTensor2.identity() * (jnp.trace(jnp.asarray(A)) / 3))
        # Construit l'entrée du réseau
        if Q is not None:
            I2 = jnp.atleast_1d(I2)
            Q  = jnp.atleast_1d(Q)
            x = jnp.concatenate([I2, Q])
        else:
            x = I2
        return super().__call__(x)

    def __call__(self, isv):
        """
        isv: objet avec
            .A : SymmetricTensor2 (forces sur alpha_p)
            .Q : jnp.ndarray (forces sur q)
        """
        A_p = isv.epsp.tensor       
        Q = isv.p       

        # Potentiel pseudo-dissipatif dual
        return  (
            self.icnn_potential(A_p, Q) - self.icnn_potential(jnp.zeros_like(A_p), jnp.zeros_like(Q))
            #  - jax.jvp(
            #      lambda a: self.icnn_potential(SymmetricTensor2(tensor=a), Q=jnp.zeros_like(Q)),
            #      (jnp.zeros_like(A_p),),
            #      (A_p,),
            #  )[1]
            #  - jax.jvp(
            #      lambda q: self.icnn_potential(SymmetricTensor2(tensor=jnp.zeros_like(A_p)), Q=q),
            #      (jnp.zeros_like(Q),),
            #      (Q,),
            #  )[1]
        )

class ICNNDissipationPotential(ICNN):
    def icnn_potential(self, A, Q=None):
        """
        A : SymmetricTensor2 (batch Nvar, 3,3)
        Q : jnp.ndarray (batch Nvar,) ou None
        """
        # Calcul invariants principaux du batch de A
        # I1, I2, I3 = jax.vmap(main_invariants)(A)  # shape (Nvar,)
        # I1, I2, I3 = main_invariants(A)
        s = A - jnp.trace(A)/3 * jnp.eye(3)
        J2 = 2./3. * jnp.sum(s * s)
        # Construit l'entrée du réseau
        if Q is not None:
            J2 = jnp.squeeze(J2)
            Q  = jnp.squeeze(Q)
            x = jnp.stack([J2, 10 * Q])
        else:
            x = J2
        return super().__call__(x)

    def __call__(self, isv):
        """
        isv: objet avec
            .A : SymmetricTensor2 (forces sur alpha_p)
            .Q : jnp.ndarray (forces sur q)
        """
        A_p = isv.epsp.tensor       
        Q = isv.p       

        # Potentiel pseudo-dissipatif dual
        return (
            self.icnn_potential(A_p, Q) - self.icnn_potential(jnp.zeros_like(A_p), jnp.zeros_like(Q))
            - jax.jvp(
                lambda a: self.icnn_potential(a, Q=jnp.zeros_like(Q)),
                (jnp.zeros_like(A_p),),
                (A_p,),
            )[1]
            - jax.jvp(
            lambda q: self.icnn_potential(jnp.zeros_like(A_p), Q=q),
            (jnp.zeros_like(Q),),
            (Q,),
        )[1]
        )


In [10]:
hidden_dims = [16,16] # (16,16,16) 6, 12, 18, 12, 6
Nvar = 1
key = jax.random.PRNGKey(42)
key1, key2, key3 = jax.random.split(key, num=3)  # define keys for reproducibility
nu = TRUE["nu"]
elasticity = jm.LinearElasticIsotropic(E=TRUE["E"] * eps_mean / sig_mean, nu=nu)

nn_anelastic = ICNN(1, hidden_dims, key1)  # one input the second main invariant
nn_hardening = ICNN(Nvar, hidden_dims, key2)#,use_quadratic_passthrough=True)  # Nvar inputs
free_energy = FreeEnergy(elasticity,nn_hardening) # nn_anelastic, nn_hardening) # 
dissipation_potential = ICNNDissipationPotential(Nvar + 1, hidden_dims,key3)#, use_quadratic_passthrough=True)

class GSM(jm.GeneralizedStandardMaterialDual):
    def make_internal_state(self):
        return InternalState()#Nvar=Nvar)


gsm = GSM(free_energy, dissipation_potential)
state0 = gsm.init_state()

In [24]:
dt_val = 0.01 * eps_mean
t_val = dt_val * jnp.arange(len(eps_path_norm)+1)
state_init      = compute_evolution(gsm, eps_path_norm, t_val)
# state_init    = compute_state_jit(gsm, eps_path_norm, dt_val)

In [25]:
state_init.internal.epsp.shape

(240, 6)

In [26]:
plt.figure()
plt.plot(state_init.internal.epsp[:,0,0], label="epsp")
plt.plot(state_init.stress[:,0,0], label="stress")
plt.plot(eps_path_norm, label="eps")
plt.legend()
plt.savefig("epsp.png")

plt.figure()
plt.plot(state_init.internal.p.squeeze(), label="p")
plt.legend()
plt.savefig("p.png")

In [27]:
plt.figure()
plt.plot(state_init.stress[:,0,0], label="gsm init")
plt.plot(sig_noisy_norm, label="data")
plt.legend()
plt.savefig('gms_init.png')

In [28]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6.  PARTITION trainable / gelé + FONCTION DE COÛT
# ═══════════════════════════════════════════════════════════════════════════════
frozen_fields = ["free_energy.elasticity"]
trainable, static = partition_by_node_names(gsm, frozen_fields)
 
 
@eqx.filter_jit
def loss_and_grad(trainable):
    def loss_fn(tr):
        mat     = eqx.combine(tr, static)
        state = compute_evolution(mat, eps_path_norm, t_val)
        sig_hat = state.stress[:,0,0]
        return jnp.mean((sig_hat - sig_noisy_norm) ** 2), sig_hat
    return eqx.filter_value_and_grad(loss_fn, has_aux=True)(trainable)
 
# Warm-up JIT
print("Compilation JIT (première passe)...")
t0 = time.time()
(loss0, _), _ = loss_and_grad(trainable)
print(f"  Compilation : {time.time()-t0:.1f} s, perte initiale = {float(loss0):.5f}")
 


Compilation JIT (première passe)...
  Compilation : 24.0 s, perte initiale = 3.93468


In [29]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7.  BOUCLE D'OPTIMISATION Adam
# ═══════════════════════════════════════════════════════════════════════════════
learning_rate = 0.5e-2
scheduler = optax.cosine_decay_schedule(learning_rate, decay_steps=150, alpha=0.01)
optimizer = optax.chain(
    optax.scale_by_adam(b1=0.9, b2=0.999),
    optax.scale_by_param_block_rms(),
    # optax.scale_by_schedule(lambda count: -scheduler(count)),
    optax.scale_by_learning_rate(learning_rate)
)
opt_state = optimizer.init(eqx.filter(trainable, eqx.is_array))
 
N_ITER    = 150
log_every = 1
patience  = 20       # arrêt si pas d'amélioration depuis N pas
tol       = 5e-6
loss_history = []
 
print(f"\nOptimisation (max {N_ITER} itérations)...")
t_start   = time.time()
current   = trainable # trainable
best_loss = float("inf")
no_improve = 0
 
for i in range(N_ITER):
    (loss_val, sig_hat), grads = loss_and_grad(current)
    lv = float(loss_val)
    loss_history.append(lv)
 
    # Early stopping
    if lv < best_loss - tol:
        best_loss  = lv
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"  → Arrêt anticipé à l'iter {i+1} (convergence)")
            break
 
    updates, opt_state = optimizer.update(
        eqx.filter(grads, eqx.is_array),
        opt_state,
        eqx.filter(current, eqx.is_array),
    )
    current = eqx.apply_updates(current, updates)
 
    if (i + 1) % log_every == 0 or i == 0:
        elapsed = time.time() - t_start
        print(f"  iter {i+1:4d} | perte = {lv:.6f} | {elapsed:.0f} s")
 
material_id = eqx.combine(current, static)
sig_id      = compute_evolution(material_id, eps_path_norm, t_val)
print(f"\nOptimisation terminée en {time.time()-t_start:.0f} s")


Optimisation (max 150 itérations)...
  iter    1 | perte = 3.934679 | 5 s
  iter    2 | perte = 3.868780 | 8 s
  iter    3 | perte = 3.803641 | 10 s
  iter    4 | perte = 3.739275 | 12 s
  iter    5 | perte = 3.675692 | 13 s
  iter    6 | perte = 3.612905 | 15 s
  iter    7 | perte = 3.550922 | 16 s
  iter    8 | perte = 3.489752 | 18 s
  iter    9 | perte = 3.429400 | 20 s
  iter   10 | perte = 3.369873 | 22 s
  iter   11 | perte = 3.311176 | 23 s
  iter   12 | perte = 3.253313 | 25 s
  iter   13 | perte = 3.196288 | 27 s
  iter   14 | perte = 3.140106 | 28 s
  iter   15 | perte = 3.084770 | 30 s
  iter   16 | perte = 3.030280 | 33 s
  iter   17 | perte = 2.976639 | 35 s
  iter   18 | perte = 2.923847 | 37 s
  iter   19 | perte = 2.871904 | 39 s
  iter   20 | perte = 2.820808 | 42 s
  iter   21 | perte = 2.770558 | 44 s
  iter   22 | perte = 2.721151 | 46 s
  iter   23 | perte = 2.672585 | 49 s
  iter   24 | perte = 2.624854 | 51 s
  iter   25 | perte = 2.577956 | 54 s
  iter   26 | 

In [44]:
plt.figure()
state_id      = compute_evolution(material_id, eps_path_norm, t_val)
plt.plot(state_id.stress[:,0,0], label="gsm trained")
plt.plot(sig_noisy_norm, label="data")
plt.legend()
plt.savefig('gms_train.png')
plt.figure()
plt.plot(state_id.internal.epsp[:,0,0], label="epsp")
plt.plot(eps_path_norm, label="eps")
plt.legend()
plt.savefig('gms_eps_train.png')
plt.figure()
plt.plot(state_id.internal.p, label="p")
plt.legend()
plt.savefig('p_id.png')